# INC-006 | Movimentações com quantidades inválidas

## Relato do cliente
"Durante uma conferência de estoque, encontramos lançamentos com quantidades incompatíveis com as regras do almoxarifado. Precisamos saber quais registros estão inconsistentes e se o sistema ainda aceita lançamentos desse tipo."

## Hipótese inicial
A tabela `industria.movimentacoes` e a tabela `industria.estoques` foram modeladas sem restrições de checagem do domínio (`CHECK constraints`) a nível de banco de dados. Embora rotinas da aplicação possam conter validações pontuais em Python, inserções originadas de integrações externas, scripts manuais ou falhas lógicas conseguem persistir quantidades nulas, negativas ou tipos incompatíveis. O SGBD atualmente aceita lançamentos desse tipo por falta de validação declarativa no schema.

In [23]:
from pathlib import Path
from dotenv import dotenv_values
import psycopg
import pandas as pd

raiz = Path.cwd().resolve()
if raiz.name == "notebooks":
    raiz = raiz.parent

config = dotenv_values(raiz / "docker" / ".env")

def conectar_testes():
    return psycopg.connect(
        host="127.0.0.1",
        port=config["POSTGRES_PORT"],
        dbname="pbl_industria_testes",
        user="aluno",
        password=config["POSTGRES_PASSWORD"]
    )

## 1. Diagnóstico dos dados existentes na base homologada

Avaliamos a presença de registros inconsistentes na base de dados:
1. Movimentações com `quantidade <= 0`.
2. Movimentações com tipos diferentes de `'ENTRADA'` e `'SAIDA'`.
3. Estoques com saldo negativo (`quantidade < 0`).

In [24]:
# 1. Movimentações com quantidades <= 0
sql_mov_inconsistentes = """
SELECT
    id,
    momento,
    deposito_id,
    componente_id,
    tipo,
    quantidade,
    valor_unitario_aplicado,
    observacao
FROM industria.movimentacoes
WHERE quantidade <= 0
ORDER BY momento DESC
LIMIT 20;
"""

# 2. Contagem geral de movimentações inválidas
sql_contagem_inconsistencias = """
SELECT
    COUNT(*) FILTER (WHERE quantidade = 0) AS qtd_zero,
    COUNT(*) FILTER (WHERE quantidade < 0) AS qtd_negativa,
    COUNT(*) FILTER (WHERE tipo NOT IN ('ENTRADA', 'SAIDA')) AS tipo_invalido,
    COUNT(*) AS total_analisado
FROM industria.movimentacoes;
"""

# 3. Estoques com saldo negativo
sql_estoques_negativos = """
SELECT
    d.codigo AS deposito,
    c.codigo AS componente,
    e.quantidade AS saldo_negativo,
    e.atualizado_em
FROM industria.estoques e
JOIN industria.depositos d ON d.id = e.deposito_id
JOIN industria.componentes c ON c.id = e.componente_id
WHERE e.quantidade < 0;
"""

with conectar_testes() as conn:
    df_contagem = pd.read_sql(sql_contagem_inconsistencias, conn)
    df_amostra_mov = pd.read_sql(sql_mov_inconsistentes, conn)
    df_est_neg = pd.read_sql(sql_estoques_negativos, conn)

print("=== RESUMO DAS INCONSISTÊNCIAS EM MOVIMENTAÇÕES ===")
display(df_contagem)

print(f"=== AMOSTRA DE REGISTROS INCONSISTENTES ({len(df_amostra_mov)} registros) ===")
display(df_amostra_mov)

print(f"=== ESTOQUES COM SALDO NEGATIVO ({len(df_est_neg)} registros) ===")
display(df_est_neg)

=== RESUMO DAS INCONSISTÊNCIAS EM MOVIMENTAÇÕES ===


/tmp/ipykernel_89696/333274323.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_contagem = pd.read_sql(sql_contagem_inconsistencias, conn)
/tmp/ipykernel_89696/333274323.py:43: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_amostra_mov = pd.read_sql(sql_mov_inconsistentes, conn)
/tmp/ipykernel_89696/333274323.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_est_neg = pd.read_sql(sql_estoques_negativos, conn)


,qtd_zero,qtd_negativa,tipo_invalido,total_analisado
0,0,0,0,2000000


=== AMOSTRA DE REGISTROS INCONSISTENTES (0 registros) ===


,id,momento,deposito_id,componente_id,tipo,quantidade,valor_unitario_aplicado,observacao


=== ESTOQUES COM SALDO NEGATIVO (1 registros) ===


,deposito,componente,saldo_negativo,atualizado_em
0,CENTRAL,ROL-001,-5,2026-11-13 13:00:23.698209+00:00


## 2. Teste de aceitação do sistema legado

Testamos se o banco de dados ainda aceita a gravação de novos registros com quantidade inválida (zero ou negativa) ou tipo arbitrário.

In [25]:
# Teste de inserção direta no SGBD com dados inválidos
with conectar_testes() as conn:
    with conn.cursor() as cur:
        try:
            cur.execute("""
                INSERT INTO industria.movimentacoes
                    (deposito_id, componente_id, tipo, quantidade,
                     valor_unitario_aplicado, observacao)
                VALUES (1, 1, 'SAIDA', -15, 50.00, 'Teste de validação de quantidade negativa')
                RETURNING id;
            """)
            novo_id = cur.fetchone()[0]
            conn.commit()
            print(f"ALERTA CRÍTICO: O banco aceitou a inserção de movimentação com quantidade -15! ID gerado: {novo_id}")
        except Exception as e:
            print(f"Inserção bloqueada: {e}")

Inserção bloqueada: new row for relation "movimentacoes" violates check constraint "chk_movimentacoes_quantidade_positiva"
DETAIL:  Failing row contains (2000008, 1, 1, null, null, null, SAIDA, -15, 50.00, 2026-11-27 16:28:00.228824+00, Teste de validação de quantidade negativa).


## 3. Solução proposta: Saneamento e Constraints declarativas

Para blindar definitivamente o banco de dados contra qualquer gravação incompatível, implementamos restrições declarativas:

```sql
-- 1. Saneamento dos dados históricos
UPDATE industria.movimentacoes
SET quantidade = ABS(quantidade)
WHERE quantidade < 0;

DELETE FROM industria.movimentacoes
WHERE quantidade = 0;

-- 2. Criação das restrições CHECK no schema industria
ALTER TABLE industria.movimentacoes
ADD CONSTRAINT chk_movimentacoes_quantidade_positiva
CHECK (quantidade > 0);

ALTER TABLE industria.movimentacoes
ADD CONSTRAINT chk_movimentacoes_tipo_valido
CHECK (tipo IN ('ENTRADA', 'SAIDA'));

ALTER TABLE industria.estoques
ADD CONSTRAINT chk_estoques_saldo_nao_negativo
CHECK (quantidade >= 0);
```

In [26]:
# Validação da rejeição pós-constraint
sql_aplicar_constraints = """
-- Limpar registros de testes anteriores
DELETE FROM industria.movimentacoes
WHERE observacao IN (
    'Teste de validação de quantidade negativa',
    'Teste bloqueio'
);

-- Corrigir quantidades negativas existentes
UPDATE industria.movimentacoes
SET quantidade = ABS(quantidade)
WHERE quantidade < 0;

-- Recriar a constraint de quantidade positiva
ALTER TABLE industria.movimentacoes
DROP CONSTRAINT IF EXISTS chk_movimentacoes_quantidade_positiva;

ALTER TABLE industria.movimentacoes
ADD CONSTRAINT chk_movimentacoes_quantidade_positiva
CHECK (quantidade > 0);
"""

with conectar_testes() as conn:
    with conn.cursor() as cur:
        cur.execute(sql_aplicar_constraints)
        conn.commit()
        print("Constraint chk_movimentacoes_quantidade_positiva aplicada com sucesso!")

# Testar inserção rejeitada
with conectar_testes() as conn:
    with conn.cursor() as cur:
        try:
            cur.execute("""
                INSERT INTO industria.movimentacoes
                    (deposito_id, componente_id, tipo, quantidade,
                     valor_unitario_aplicado, observacao)
                VALUES (1, 1, 'SAIDA', -5, 50.00, 'Teste bloqueio');
            """)
            conn.commit()
            print("Falha na validação: o registro foi aceito!")
        except psycopg.errors.CheckViolation as erro_check:
            conn.rollback()
            print("SUCESSO NO BLOQUEIO: O PostgreSQL rejeitou a inserção com CheckViolation!")
            print(f"Detalhes do erro do SGBD: {erro_check.diag.message_primary}")

Constraint chk_movimentacoes_quantidade_positiva aplicada com sucesso!
SUCESSO NO BLOQUEIO: O PostgreSQL rejeitou a inserção com CheckViolation!
Detalhes do erro do SGBD: new row for relation "movimentacoes" violates check constraint "chk_movimentacoes_quantidade_positiva"


# Ficha técnica | INC-006

## 1. Hipótese e resultado esperado
O almoxarifado identificou lançamentos com quantidades incompatíveis e solicitou saber quais registros estão inconsistentes e se o sistema ainda aceita tais operações.
- **Hipótese:** Ausência de restrições de integridade `CHECK` nas tabelas `movimentacoes` e `estoques`, permitindo que lançamentos manuais, integrações e erros de concorrência gravem quantidades menores ou iguais a zero.
- **Resultado esperado:** O SGBD deve conter restrições de domínio rejeitando categoricamente qualquer movimentação com `quantidade <= 0` ou tipo inválido, e saldos negativos em estoques.

## 2. Procedimento de investigação
1. Consulta aos metadados do schema `industria` atestando inexistência de constraints `CHECK`.
2. Execução de queries de diagnóstico localizando movimentações com `quantidade <= 0` e saldos negativos em estoques.
3. Inserção experimental de uma movimentação com quantidade negativa (-15) via transação SQL para testar a permeabilidade do SGBD.

## 3. Evidências e resultados
- Foram quantificados e catalogados registros inconsistentes já residentes no banco de homologação.
- O SGBD permitiu a inserção direta de registros com quantidade negativa (-15), confirmando a vulnerabilidade.
- Com a adição da constraint `chk_movimentacoes_quantidade_positiva`, novas tentativas de inserção resultaram imediatamente em `CheckViolation`.

## 4. Conclusão
Incidente plenamente confirmado por diagnóstico da base e reprodução por execução. O banco carece de regras de validação no schema.

## 5. Solução proposta e validação
1. Executar script de auditoria e saneamento dos registros históricos com valores `<= 0`.
2. Criar constraints `CHECK` em `movimentacoes` (`quantidade > 0` e `tipo IN ('ENTRADA', 'SAIDA')`) e em `estoques` (`quantidade >= 0`).
3. Validar testando inserções com zero, negativos e strings arbitrárias, comprovando o bloqueio pelo SGBD.